# Experiment 4: Forward Chaining Inference Engine (Data-Driven Reasoning)

## Aim
To implement a forward chaining inference engine and use it to identify an animal from observed features, in the manner of the classic ZOOKEEPER system.

## Theory
Forward chaining starts from the available data and applies rules to derive new facts until either a goal is reached or no rule can fire. It is **data-driven**: reasoning is triggered by what is known, not by what is asked. This suits monitoring, configuration and interpretation tasks where many conclusions may follow from the same data.

Each cycle re-computes the applicable rules against the growing fact base and fires one of them, adding its conclusion to the facts. The chain of fired rules (e.g. R1 -> R4 -> R6) is itself valuable: it is the raw material for the explanation facility. Intermediate conclusions such as *is mammal* and *is carnivore* show how rule chaining builds abstraction level by level.

## Algorithm
1. Initialise the fact set with the observed features.
2. Repeat: find all rules whose conditions are satisfied and whose conclusion is **new**.
3. If none exists, stop; otherwise fire the first such rule, add its conclusion, and record its identifier.
4. On termination, report the final fact set, the firing sequence, and the identification.

## Knowledge Base (Production Rules)

| Rule | Conditions | Conclusion |
|------|-----------|------------|
| R1 | has hair | is mammal |
| R2 | gives milk | is mammal |
| R3 | has feathers | is bird |
| R4 | is mammal, eats meat | is carnivore |
| R5 | is mammal, has pointed teeth, has claws, has forward-pointing eyes | is carnivore |
| R6 | is mammal, is carnivore, has tawny color, has dark spots | is cheetah |
| R7 | is mammal, is carnivore, has tawny color, has black stripes | is tiger |
| R8 | is mammal, has long neck, has long legs, has dark spots | is giraffe |
| R9 | is mammal, has black stripes, NOT is carnivore | is zebra |
| R10 | is bird, does not fly, has long neck, has long legs | is ostrich |

## Procedure
1. Define the animal-identification knowledge base using IF-THEN production rules.
2. Initialise the working memory (WM) with the given observed features.
3. Match the conditions of each rule with the facts available in WM.
4. Construct the conflict set containing all applicable rules whose conclusions are not already present in WM.
5. Select the first rule from the conflict set according to the given rule order.
6. Fire the selected rule and add its conclusion to WM.
7. Record the identifier of the fired rule.
8. Repeat the matching and rule-firing process with the updated WM.
9. Continue until the required animal is identified or the conflict set becomes empty.
10. Display the final working memory, firing sequence, and identified animal.

## Implementation

In [1]:
# Knowledge Base Definition
# Each rule:
#   'id'         - rule identifier
#   'conditions' - all must be present in WM
#   'neg_conds'  - none must be present in WM (negation-as-failure)
#   'conclusion' - fact added to WM when fired

RULES = [
    # Mammal classification
    {'id': 'R1',  'conditions': ['has hair'],
     'neg_conds': [], 'conclusion': 'is mammal'},
    {'id': 'R2',  'conditions': ['gives milk'],
     'neg_conds': [], 'conclusion': 'is mammal'},
    # Bird classification
    {'id': 'R3',  'conditions': ['has feathers'],
     'neg_conds': [], 'conclusion': 'is bird'},
    # Carnivore classification
    {'id': 'R4',  'conditions': ['is mammal', 'eats meat'],
     'neg_conds': [], 'conclusion': 'is carnivore'},
    {'id': 'R5',  'conditions': ['is mammal', 'has pointed teeth',
                                  'has claws', 'has forward-pointing eyes'],
     'neg_conds': [], 'conclusion': 'is carnivore'},
    # Animal identification
    {'id': 'R6',  'conditions': ['is mammal', 'is carnivore',
                                  'has tawny color', 'has dark spots'],
     'neg_conds': [], 'conclusion': 'is cheetah'},
    # Exercise 1 - tiger
    {'id': 'R7',  'conditions': ['is mammal', 'is carnivore',
                                  'has tawny color', 'has black stripes'],
     'neg_conds': [], 'conclusion': 'is tiger'},
    # Exercise 1 - giraffe (long neck, long legs)
    {'id': 'R8',  'conditions': ['is mammal', 'has long neck',
                                  'has long legs', 'has dark spots'],
     'neg_conds': [], 'conclusion': 'is giraffe'},
    # Exercise 1 - zebra (black stripes, not carnivore)
    {'id': 'R9',  'conditions': ['is mammal', 'has black stripes'],
     'neg_conds': ['is carnivore'], 'conclusion': 'is zebra'},
    # Ostrich (bird rule)
    {'id': 'R10', 'conditions': ['is bird', 'does not fly',
                                  'has long neck', 'has long legs'],
     'neg_conds': [], 'conclusion': 'is ostrich'},
]

ANIMAL_FACTS = {'is cheetah', 'is tiger', 'is giraffe', 'is zebra', 'is ostrich'}

print('Knowledge base loaded:', len(RULES), 'rules')

Knowledge base loaded: 10 rules


In [2]:
# Forward Chaining Engine - fires ONE rule per cycle (first applicable)

def forward_chain(initial_facts, rules=RULES, animal_facts=ANIMAL_FACTS, verbose=True):
    # Run the forward chaining inference engine.
    # Returns: (wm, firing_sequence, identified_animal)
    wm = set(initial_facts)   # working memory
    firing_sequence = []
    cycle = 0

    if verbose:
        print('=' * 60)
        print('FORWARD CHAINING INFERENCE ENGINE')
        print('=' * 60)
        print('Initial WM :', sorted(wm))
        print()

    while True:
        cycle += 1

        # Match phase: build conflict set
        conflict_set = [
            r for r in rules
            if all(c in wm for c in r['conditions'])      # all positives present
            and all(n not in wm for n in r['neg_conds'])  # all negatives absent
            and r['conclusion'] not in wm                 # conclusion is new
        ]

        if verbose:
            print('Cycle', cycle, ':')
            cs_ids = [r['id'] for r in conflict_set]
            print('  Conflict set :', cs_ids if cs_ids else 'empty')

        if not conflict_set:
            if verbose:
                print('  No applicable rule - inference halted.')
            break

        # Select phase: pick first rule
        selected = conflict_set[0]

        # Execute phase: fire the rule
        wm.add(selected['conclusion'])
        firing_sequence.append(selected['id'])

        if verbose:
            print('  Selected rule:', selected['id'])
            print('  New fact     :', repr(selected['conclusion']))
            print('  WM now       :', sorted(wm))
            print()

        # Stop early if an animal has been identified
        if selected['conclusion'] in animal_facts:
            break

    identified = next((f for f in wm if f in animal_facts), None)

    if verbose:
        print('=' * 60)
        print('RESULTS')
        print('=' * 60)
        seq_str = ' -> '.join(firing_sequence) if firing_sequence else 'none'
        print('Firing sequence  :', seq_str)
        print('Final WM         :', sorted(wm))
        id_str = identified if identified else 'No identification possible'
        print('Identified animal:', id_str)
        print()

    return wm, firing_sequence, identified


print('Engine defined.')

Engine defined.


## Observations - Base Case: Identifying a Cheetah

Initial facts: `has hair`, `eats meat`, `has tawny color`, `has dark spots`

Expected firing sequence: **R1 -> R4 -> R6**

In [3]:
# Base case: Cheetah
cheetah_facts = ['has hair', 'eats meat', 'has tawny color', 'has dark spots']
wm, seq, animal = forward_chain(cheetah_facts)

FORWARD CHAINING INFERENCE ENGINE
Initial WM : ['eats meat', 'has dark spots', 'has hair', 'has tawny color']

Cycle 1 :
  Conflict set : ['R1']
  Selected rule: R1
  New fact     : 'is mammal'
  WM now       : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is mammal']

Cycle 2 :
  Conflict set : ['R4']
  Selected rule: R4
  New fact     : 'is carnivore'
  WM now       : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is carnivore', 'is mammal']

Cycle 3 :
  Conflict set : ['R6']
  Selected rule: R6
  New fact     : 'is cheetah'
  WM now       : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is carnivore', 'is cheetah', 'is mammal']

RESULTS
Firing sequence  : R1 -> R4 -> R6
Final WM         : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is carnivore', 'is cheetah', 'is mammal']
Identified animal: is cheetah



## Exercise 1 - Identify Tiger, Giraffe, and Zebra

Rules R7 (tiger), R8 (giraffe), and R9 (zebra) have been added to the knowledge base.

- **Tiger**: mammal, eats meat, tawny color, black stripes
- **Giraffe**: mammal, long neck, long legs, dark spots
- **Zebra**: mammal, black stripes, **not** carnivore (negation-as-failure via R9)

In [4]:
# Exercise 1a: Tiger
print('-' * 30 + ' TIGER ' + '-' * 30)
tiger_facts = ['has hair', 'eats meat', 'has tawny color', 'has black stripes']
forward_chain(tiger_facts);

------------------------------ TIGER ------------------------------
FORWARD CHAINING INFERENCE ENGINE
Initial WM : ['eats meat', 'has black stripes', 'has hair', 'has tawny color']

Cycle 1 :
  Conflict set : ['R1']
  Selected rule: R1
  New fact     : 'is mammal'
  WM now       : ['eats meat', 'has black stripes', 'has hair', 'has tawny color', 'is mammal']

Cycle 2 :
  Conflict set : ['R4', 'R9']
  Selected rule: R4
  New fact     : 'is carnivore'
  WM now       : ['eats meat', 'has black stripes', 'has hair', 'has tawny color', 'is carnivore', 'is mammal']

Cycle 3 :
  Conflict set : ['R7']
  Selected rule: R7
  New fact     : 'is tiger'
  WM now       : ['eats meat', 'has black stripes', 'has hair', 'has tawny color', 'is carnivore', 'is mammal', 'is tiger']

RESULTS
Firing sequence  : R1 -> R4 -> R7
Final WM         : ['eats meat', 'has black stripes', 'has hair', 'has tawny color', 'is carnivore', 'is mammal', 'is tiger']
Identified animal: is tiger



In [5]:
# Exercise 1b: Giraffe
print('-' * 28 + ' GIRAFFE ' + '-' * 28)
giraffe_facts = ['has hair', 'has long neck', 'has long legs', 'has dark spots']
forward_chain(giraffe_facts);

---------------------------- GIRAFFE ----------------------------
FORWARD CHAINING INFERENCE ENGINE
Initial WM : ['has dark spots', 'has hair', 'has long legs', 'has long neck']

Cycle 1 :
  Conflict set : ['R1']
  Selected rule: R1
  New fact     : 'is mammal'
  WM now       : ['has dark spots', 'has hair', 'has long legs', 'has long neck', 'is mammal']

Cycle 2 :
  Conflict set : ['R8']
  Selected rule: R8
  New fact     : 'is giraffe'
  WM now       : ['has dark spots', 'has hair', 'has long legs', 'has long neck', 'is giraffe', 'is mammal']

RESULTS
Firing sequence  : R1 -> R8
Final WM         : ['has dark spots', 'has hair', 'has long legs', 'has long neck', 'is giraffe', 'is mammal']
Identified animal: is giraffe



In [6]:
# Exercise 1c: Zebra
# No 'eats meat' -> R4 cannot fire -> 'is carnivore' never derived
# R9 neg_cond 'is carnivore' is NOT violated -> R9 fires
print('-' * 29 + ' ZEBRA ' + '-' * 29)
zebra_facts = ['has hair', 'has black stripes']
forward_chain(zebra_facts);

----------------------------- ZEBRA -----------------------------
FORWARD CHAINING INFERENCE ENGINE
Initial WM : ['has black stripes', 'has hair']

Cycle 1 :
  Conflict set : ['R1']
  Selected rule: R1
  New fact     : 'is mammal'
  WM now       : ['has black stripes', 'has hair', 'is mammal']

Cycle 2 :
  Conflict set : ['R9']
  Selected rule: R9
  New fact     : 'is zebra'
  WM now       : ['has black stripes', 'has hair', 'is mammal', 'is zebra']

RESULTS
Firing sequence  : R1 -> R9
Final WM         : ['has black stripes', 'has hair', 'is mammal', 'is zebra']
Identified animal: is zebra



## Exercise 2 - Facts That Satisfy No Identification Rule

Provide facts that let the engine derive an intermediate class (e.g. *is mammal*, *is carnivore*) but never reach any specific animal. The engine should report the most specific class it could establish.

In [7]:
# Exercise 2
# 'has hair'  -> fires R1 -> 'is mammal'
# 'eats meat' -> fires R4 -> 'is carnivore'
# No color / stripe / spot facts -> no animal rule can complete -> halts

def forward_chain_most_specific(initial_facts, rules=RULES, animal_facts=ANIMAL_FACTS):
    # Same as forward_chain but reports the most specific intermediate class on failure.
    # Class hierarchy (most specific first)
    class_hierarchy = ['is carnivore', 'is mammal', 'is bird']

    wm, seq, identified = forward_chain(initial_facts, rules, animal_facts, verbose=True)

    if identified is None:
        most_specific = next((c for c in class_hierarchy if c in wm), None)
        print('-' * 60)
        print('Exercise 2 - No animal identified.')
        ms_str = most_specific if most_specific else 'none (no class derived)'
        print('Most specific class established:', ms_str)
        print()
    return wm, seq, identified


print('=' * 60)
print('EXERCISE 2 - No identification possible')
print('=' * 60)
partial_facts = ['has hair', 'eats meat']
forward_chain_most_specific(partial_facts);

EXERCISE 2 - No identification possible
FORWARD CHAINING INFERENCE ENGINE
Initial WM : ['eats meat', 'has hair']

Cycle 1 :
  Conflict set : ['R1']
  Selected rule: R1
  New fact     : 'is mammal'
  WM now       : ['eats meat', 'has hair', 'is mammal']

Cycle 2 :
  Conflict set : ['R4']
  Selected rule: R4
  New fact     : 'is carnivore'
  WM now       : ['eats meat', 'has hair', 'is carnivore', 'is mammal']

Cycle 3 :
  Conflict set : empty
  No applicable rule - inference halted.
RESULTS
Firing sequence  : R1 -> R4
Final WM         : ['eats meat', 'has hair', 'is carnivore', 'is mammal']
Identified animal: No identification possible

------------------------------------------------------------
Exercise 2 - No animal identified.
Most specific class established: is carnivore



## Exercise 3 - Fire ALL Applicable Rules per Cycle

Modify the engine so that **every** rule in the conflict set fires in a single cycle (breadth-first), then compare the firing sequence with the one-rule-per-cycle version.

In [8]:
# Exercise 3 - All-rules engine

def forward_chain_all(initial_facts, rules=RULES, animal_facts=ANIMAL_FACTS, verbose=True):
    # Forward chaining engine that fires ALL applicable rules each cycle.
    wm = set(initial_facts)
    firing_sequence = []
    cycle = 0

    if verbose:
        print('=' * 60)
        print('FORWARD CHAINING ENGINE - ALL RULES PER CYCLE')
        print('=' * 60)
        print('Initial WM :', sorted(wm))
        print()

    while True:
        cycle += 1

        conflict_set = [
            r for r in rules
            if all(c in wm for c in r['conditions'])
            and all(n not in wm for n in r['neg_conds'])
            and r['conclusion'] not in wm
        ]

        if verbose:
            print('Cycle', cycle, ':')
            cs_ids = [r['id'] for r in conflict_set]
            print('  Conflict set :', cs_ids if cs_ids else 'empty')

        if not conflict_set:
            if verbose:
                print('  No applicable rule - inference halted.')
            break

        # Fire ALL rules in the conflict set
        new_facts = set()
        for r in conflict_set:
            new_facts.add(r['conclusion'])
            firing_sequence.append(r['id'])

        wm |= new_facts

        if verbose:
            fired_ids = [r['id'] for r in conflict_set]
            print('  Fired rules  :', fired_ids)
            print('  New facts    :', sorted(new_facts))
            print('  WM now       :', sorted(wm))
            print()

        if new_facts & animal_facts:
            break

    identified = next((f for f in wm if f in animal_facts), None)

    if verbose:
        print('=' * 60)
        print('RESULTS')
        print('=' * 60)
        seq_str = ' -> '.join(firing_sequence) if firing_sequence else 'none'
        print('Firing sequence  :', seq_str)
        print('Final WM         :', sorted(wm))
        id_str = identified if identified else 'No identification possible'
        print('Identified animal:', id_str)
        print()

    return wm, firing_sequence, identified


print('All-rules engine defined.')

All-rules engine defined.


In [9]:
# Comparison: one-rule vs all-rules for the cheetah scenario
print('ONE RULE PER CYCLE')
print('=' * 40)
_, seq_one, _ = forward_chain(cheetah_facts, verbose=False)
print('Firing sequence  :', ' -> '.join(seq_one))
print('Total rules fired:', len(seq_one))
print()

print('ALL RULES PER CYCLE')
print('=' * 40)
_, seq_all, _ = forward_chain_all(cheetah_facts, verbose=False)
print('Firing sequence  :', ' -> '.join(seq_all))
print('Total rules fired:', len(seq_all))
print()

print('-' * 60)
print('Observation:')
print('  One-rule : fires one rule per cycle -> more cycles, sequential.')
print('  All-rules: fires all applicable rules per cycle -> fewer cycles,')
print('             may derive multiple facts simultaneously,')
print('             final identification is the same.')

ONE RULE PER CYCLE
Firing sequence  : R1 -> R4 -> R6
Total rules fired: 3

ALL RULES PER CYCLE
Firing sequence  : R1 -> R4 -> R6
Total rules fired: 3

------------------------------------------------------------
Observation:
  One-rule : fires one rule per cycle -> more cycles, sequential.
  All-rules: fires all applicable rules per cycle -> fewer cycles,
             may derive multiple facts simultaneously,
             final identification is the same.


In [10]:
# Detailed trace for all-rules engine (cheetah)
print('Detailed trace - ALL RULES PER CYCLE (cheetah)')
forward_chain_all(cheetah_facts);

Detailed trace - ALL RULES PER CYCLE (cheetah)
FORWARD CHAINING ENGINE - ALL RULES PER CYCLE
Initial WM : ['eats meat', 'has dark spots', 'has hair', 'has tawny color']

Cycle 1 :
  Conflict set : ['R1']
  Fired rules  : ['R1']
  New facts    : ['is mammal']
  WM now       : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is mammal']

Cycle 2 :
  Conflict set : ['R4']
  Fired rules  : ['R4']
  New facts    : ['is carnivore']
  WM now       : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is carnivore', 'is mammal']

Cycle 3 :
  Conflict set : ['R6']
  Fired rules  : ['R6']
  New facts    : ['is cheetah']
  WM now       : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is carnivore', 'is cheetah', 'is mammal']

RESULTS
Firing sequence  : R1 -> R4 -> R6
Final WM         : ['eats meat', 'has dark spots', 'has hair', 'has tawny color', 'is carnivore', 'is cheetah', 'is mammal']
Identified animal: is cheetah



## Conclusion

The forward chaining inference engine successfully identifies animals from observed features by iteratively applying IF-THEN production rules to an expanding working memory.

- **Base case**: Facts `{has hair, eats meat, has tawny color, has dark spots}` fire rules **R1 -> R4 -> R6**, identifying a *cheetah*.
- **Exercise 1**: Extended rules R7, R8, R9 correctly identify a *tiger*, *giraffe*, and *zebra*. R9 uses negation-as-failure (zebra must **not** be a carnivore).
- **Exercise 2**: When facts are insufficient for full identification, the engine reports the most specific intermediate class derived (e.g. *is carnivore*).
- **Exercise 3**: Firing all applicable rules per cycle reduces the number of inference cycles while producing the same final conclusion; the firing sequence is compressed into simultaneous batches instead of sequential steps.